In [16]:
%pip install duckdb datashader colorcet plotly pandas pillow geopandas shapely pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.5/32.5 MB 75.3 MB/s  0:00:006m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 73.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [geopandas]/4 [geopandas]
Note: you may need to restart the kernel to use updated packages.


In [52]:
import duckdb
import datashader as ds
import datashader.transfer_functions as tf
import colorcet as cc
import os
import plotly.express as px
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

In [53]:
GFW_FROM, GFW_TO = 2013, 2013
OBIS_FROM, OBIS_TO = 2013, 2013
OBIS_SPEC = "*"

In [54]:
years_gfw = range(GFW_FROM, GFW_TO + 1)
paths_gfw = "/mnt/shared_data/finflow/gfw_raw_deprecated/2025/*.parquet"
path_obis = f"/mnt/shared_data/finflow/obis_raw/{OBIS_SPEC}/*/*.parquet"

query_gfw = f"SELECT lon, lat, hours FROM read_parquet('{paths_gfw}')"

query_obis = f"""
SELECT decimalLongitude as lon, decimalLatitude as lat 
FROM read_parquet('{path_obis}') 
WHERE year(eventDate) BETWEEN {OBIS_FROM} AND {OBIS_TO}
  AND lon IS NOT NULL AND lat IS NOT NULL
"""

In [ ]:
df_gfw = duckdb.query(query_gfw).to_df()
df_obis = duckdb.query(query_obis).to_df()

W, H = 6000, 3000
cvs = ds.Canvas(plot_width=W, plot_height=H, x_range=(-180, 180), y_range=(-90, 90))

base_map = Image.open("/mnt/shared_data/finflow/images/base_map.png").resize((W, H)).convert("RGBA")
#base_map = Image.new("RGBA", (W, H), (0, 0, 0, 255))

agg_gfw = cvs.points(df_gfw, 'lon', 'lat', ds.sum('hours'))
img_gfw = tf.shade(agg_gfw, cmap=cc.fire, how='log').to_pil().convert("RGBA")

agg_obis = cvs.points(df_obis, 'lon', 'lat', ds.count())
greens = ["#e0ffe0", "#90ee90", "#32cd32", "#00ff00", "#adff2f"]
img_obis = tf.shade(agg_obis, cmap=greens, how='log').to_pil().convert("RGBA")

combined = Image.alpha_composite(base_map, img_gfw)
combined = Image.alpha_composite(combined, img_obis)

fig = px.imshow(combined)

fig.update_layout(
    height=500,
    margin=dict(l=0, r=0, t=0, b=0),
    paper_bgcolor="black",
    plot_bgcolor="black",
    xaxis_visible=False,
    yaxis_visible=False,
    yaxis_scaleanchor="x"
)

fig.show(config={'scrollZoom': True})